# GSGP-NN: Test Suite & Fair Comparison

This notebook:
1. **Executes all unit tests** using `pytest`.
2. **Compares two approaches** on the Diabetes dataset with identical config:
   - `GSGPNNRegressor` (sklearn interface — GP + neural network)
   - `SimpleNN` (standard MLP baseline)

## 1. Execute Project Tests

In [1]:
import sys, os, pytest

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Running pytest on the 'tests' directory...")
exit_code = pytest.main(["-v", "../tests"])
print(f"Pytest finished with exit code: {exit_code} (0 means all tests passed)")

Running pytest on the 'tests' directory...
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0 -- /home/gallobota/miniconda3/envs/gsgp_nn/bin/python
cachedir: .pytest_cache
rootdir: /home/gallobota/GSGP_NN
plugins: anyio-4.14.1
collecting ... collected 0 items / 7 errors

==================================== ERRORS ====================================
____________________ ERROR collecting tests/test_dataset.py ____________________
ImportError while importing test module '/home/gallobota/GSGP_NN/tests/test_dataset.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
../../miniconda3/envs/gsgp_nn/lib/python3.12/importlib/__init__.py:90: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
../tests/test_dataset.py:5: in <module>
    import torch
../../miniconda3/envs/gsgp_nn/lib/

## 2. Shared Configuration & Data

Both models share the same hyperparameters so the comparison is fair.

In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

from src.utils.device import get_device

# ── Dataset ─────────────────────────────────────────────────────────
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test : {X_test.shape[0]} samples")

# ── Device ──────────────────────────────────────────────────────────
device = get_device()

# ── Shared hyperparameters ─────────────────────────────────────────
NUM_NODES = 10          # N — number of GP semantic nodes
NUM_LAYERS = 3          # K — number of GSGP-NN layers
POP_SIZE = 40           # GP population size
LR = 0.01               # learning rate
EPOCHS = 200            # max epochs
ES_PATIENCE = 20        # early stopping patience
ES_WARMUP = 10          # early stopping warm-up
RANDOM_STATE = 42       # reproducibility seed

# ── Scaled tensors (for SimpleNN) ──────────────────────────────────
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s = scaler_X.transform(X_test)
y_train_s = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_s = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

X_train_t = torch.tensor(X_train_s, dtype=torch.float32, device=device)
X_test_t  = torch.tensor(X_test_s,  dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32, device=device)
y_test_t  = torch.tensor(y_test_s,  dtype=torch.float32, device=device)

ImportError: /home/gallobota/miniconda3/envs/gsgp_nn/lib/python3.12/site-packages/torch/lib/libtorch_cpu.so: undefined symbol: iJIT_NotifyEvent

---
## 3. GSGPNNRegressor (scikit-learn Interface)

Wraps the full GP generation + semantic evaluation + neural network training in a single `fit`/`predict` call.

In [ ]:
from src.models.gsgp_nn_estimator import GSGPNNRegressor, model, complexity

gsgp_est = GSGPNNRegressor(
    num_nodes=NUM_NODES,
    num_layers=NUM_LAYERS,
    population_size=POP_SIZE,
    learning_rate=LR,
    epochs=EPOCHS,
    early_stopping_patience=ES_PATIENCE,
    early_stopping_warmup=ES_WARMUP,
    verbose=True,
    random_state=RANDOM_STATE,
)

# GSGPNNRegressor scales X and y internally — pass raw data
print("Training GSGPNNRegressor...")
gsgp_est.fit(X_train, y_train)

y_pred_gsgp = gsgp_est.predict(X_test)
mse_gsgp = mean_squared_error(y_test, y_pred_gsgp)
r2_gsgp = r2_score(y_test, y_pred_gsgp)
params_gsgp = gsgp_est.model_.count_parameters()

print(f"\nGSGPNNRegressor  |  MSE: {mse_gsgp:.2f}  |  R²: {r2_gsgp:.4f}  |  Params: {params_gsgp:,}")
print(f"SymPy expression : {model(gsgp_est)}")
print(f"Complexity       : {complexity(gsgp_est)} nodes")

---
## 4. SimpleNN (MLP Baseline)

Standard feed-forward network with approximately the same number of parameters as GSGP-NN.

In [ ]:
import torch.nn as nn
from src.training.early_stopping import EarlyStopping
from src.utils.seed import set_seed


class SimpleNN(nn.Module):
    """MLP with comparable capacity to GSGP-NN(N=10, K=3 → 401 params)."""
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 20),
            nn.ReLU(),
            nn.Linear(20, 10),
            nn.ReLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


set_seed(RANDOM_STATE)
nn_model = SimpleNN(input_dim=X_train_s.shape[1]).to(device)
params_nn = nn_model.count_parameters()

optimizer = torch.optim.Adam(nn_model.parameters(), lr=LR)
es = EarlyStopping(patience=ES_PATIENCE, min_delta=1e-5, warmup_epochs=ES_WARMUP)

loss_history_nn = []
nn_model.train()

for epoch in range(1, EPOCHS + 1):
    optimizer.zero_grad()
    y_pred = nn_model(X_train_t)
    loss = nn.functional.mse_loss(y_pred, y_train_t)
    loss.backward()
    optimizer.step()

    loss_val = loss.item()
    loss_history_nn.append(loss_val)

    if epoch <= 10 or epoch % 25 == 0:
        print(f"Epoch {epoch:3d}  |  Loss = {loss_val:.6f}")

    if es(loss_val):
        print(f"[EarlyStopping] Stopped at epoch {epoch}")
        break

print(f"\nSimpleNN final loss: {loss_history_nn[-1]:.6f}")

In [ ]:
# Evaluate SimpleNN on test set
nn_model.eval()
with torch.no_grad():
    y_pred_nn_scaled = nn_model(X_test_t).cpu().numpy()
y_pred_nn = scaler_y.inverse_transform(y_pred_nn_scaled.reshape(-1, 1)).flatten()

mse_nn = mean_squared_error(y_test, y_pred_nn)
r2_nn = r2_score(y_test, y_pred_nn)

print(f"SimpleNN           |  MSE: {mse_nn:.2f}  |  R²: {r2_nn:.4f}  |  Params: {params_nn:,}")

---
## 5. Final Comparison

Both models trained with **identical** learning rate, max epochs, early stopping policy, and random seed.

In [ ]:
print("=" * 72)
print("FAIR COMPARISON  —  Diabetes dataset")
print("=" * 72)
print(f"{'':<30} {'GSGPNNRegressor':<20} {'SimpleNN':<20}")
print(f"{'-'*30} {'-'*20} {'-'*20}")
print(f"{'Test MSE':<30} {mse_gsgp:<20.2f} {mse_nn:<20.2f}")
print(f"{'Test R²':<30} {r2_gsgp:<20.4f} {r2_nn:<20.4f}")
print(f"{'Trainable parameters':<30} {params_gsgp:<20,} {params_nn:<20,}")
print(f"{'Epochs trained':<30} {gsgp_est.history_['epochs_trained']:<20} {len(loss_history_nn):<20}")
print(f"{'Learning rate':<30} {LR:<20} {LR:<20}")
print(f"{'Early stopping patience':<30} {ES_PATIENCE:<20} {ES_PATIENCE:<20}")
print(f"{'Random seed':<30} {RANDOM_STATE:<20} {RANDOM_STATE:<20}")

print(f"\n--- SRBench ---")
print(f"SymPy expression: {model(gsgp_est)}")
print(f"Complexity      : {complexity(gsgp_est)} nodes")
print(f"{'=' * 72}")

if r2_gsgp > r2_nn:
    print(f"\n✓ GSGPNNRegressor outperforms SimpleNN by {abs(r2_gsgp - r2_nn):.4f} R²")
elif r2_nn > r2_gsgp:
    print(f"\n✗ SimpleNN outperforms GSGPNNRegressor by {abs(r2_nn - r2_gsgp):.4f} R²")
else:
    print("\n— Both models achieved the same R²")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Loss curves ────────────────────────────────────────────────────
ax = axes[0]
ax.plot(gsgp_est.history_["loss_history"], label="GSGPNNRegressor", color="green", lw=1.5)
ax.plot(loss_history_nn, label="SimpleNN", color="navy", lw=1.5, linestyle="--")
ax.set_title("Training Loss (MSE, scaled)")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.legend()
ax.grid(True, alpha=0.3)

# ── Bar chart ──────────────────────────────────────────────────────
ax = axes[1]
labels = ['GSGPNNRegressor', 'SimpleNN']
x = np.arange(len(labels))
width = 0.30

bars1 = ax.bar(x - width/2, [mse_gsgp, mse_nn], width,
               label='MSE', color=['green', 'navy'])
ax.set_ylabel('MSE')
ax.set_xticks(x)
ax.set_xticklabels(labels)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(mse_gsgp, mse_nn)*0.02,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=9)

ax2 = ax.twinx()
bars2 = ax2.bar(x + width/2, [r2_gsgp, r2_nn], width,
                label='R²', color=['lightgreen', 'lightblue'], alpha=0.7)
ax2.set_ylabel('R²')
ax2.set_ylim(0, 1)
for bar in bars2:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

ax.set_title("Test Set Metrics")
fig.tight_layout()
plt.show()